In [2]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import numpy as np
import optuna
import optuna.visualization as vis

# Add project root directory to path
sys.path.append(os.path.abspath(".."))

from src.data import load_raw_dataset
from src.features import build_feature_pipeline
from src.tune import objective

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# Load sample dataset for tuning (100,000 rows balances speed and statistical accuracy)
df_raw = load_raw_dataset(data_dir="../data/raw", split="train", nrows=100000)

# Build features
df_fe = build_feature_pipeline(df_raw)

# Separate features and target
drop_cols = ["TransactionID", "TransactionDT", "isFraud", "uid1", "uid2"]
feature_cols = [c for c in df_fe.columns if c not in drop_cols]

X = df_fe[feature_cols].copy()
y = df_fe["isFraud"].values

# Ensure categorical data types are preserved for LightGBM
cat_cols = X.select_dtypes(include=["object", "string"]).columns
for c in cat_cols:
    X[c] = X[c].astype("category")

print(f"Dataset ready for tuning: {X.shape[0]} rows, {X.shape[1]} features.")

Loading train_transaction.csv...
Loading train_identity.csv...
Performing left join on TransactionID...
Optimizing memory footprint...
Memory usage decreased to 176.91 MB (46.6% reduction)
Starting feature engineering pipeline...
Feature engineering complete. Total columns: 453
Dataset ready for tuning: 100000 rows, 448 features.


In [ ]:
# Create Optuna study to maximize out-of-fold ROC-AUC
study = optuna.create_study(
    study_name="lgb_fraud_optimization",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

# Run optimization across 30 trials
print("Starting hyperparameter optimization...")
study.optimize(
    lambda trial: objective(trial, X, y, drop_cols), 
    n_trials=30,
    show_progress_bar=True
)

print("\nBest CV ROC-AUC Score achieved:", round(study.best_value, 5))

Starting hyperparameter optimization...


  0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
# Plot optimization history (tracks how CV ROC-AUC improved over trials)
fig_history = vis.plot_optimization_history(study)
fig_history.update_layout(title="Optuna Convergence History (ROC-AUC vs. Trial)")
fig_history.show()

In [ ]:
# Plot relative importance of each hyperparameter on model performance
fig_importance = vis.plot_param_importances(study)
fig_importance.update_layout(title="Hyperparameter Importance Analysis")
fig_importance.show()

In [ ]:
# Display optimal parameters to update src/models.py
best_params = study.best_params

print("=" * 50)
print("         WINNING HYPERPARAMETERS               ")
print("=" * 50)
for param_name, param_val in best_params.items():
    print(f"{param_name:20s} : {param_val}")
print("=" * 50)